## Scenario: A software company wants to build its first AI assistant. 
### Tasks: Create a basic LLM-powered agent capable of handling user queries, maintaining context, and performing task execution workflows.

In [33]:
import os
import time
from pathlib import Path

from langchain_ollama import ChatOllama, OllamaEmbeddings

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate

from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
    ToolMessage
)

from langchain.agents import create_agent

from langgraph.checkpoint.memory import InMemorySaver

print("All libraries imported successfully!")

C:\Users\smart\AppData\Local\Temp\ipykernel_7368\3616398710.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


All libraries imported successfully!


In [35]:
PDF_PATH = "who.pdf"

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

print(f"PDF loaded successfully: {len(documents)} pages")

PDF loaded successfully: 12 pages


In [36]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks: {len(chunks)}")

Total chunks: 55


In [37]:
for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i + 1}")
    print(chunk.page_content)
    print("-" * 80)


Chunk 1
Patient safety rights charter
--------------------------------------------------------------------------------

Chunk 2
| 2 |
Patient safety rights 
10
Right to timely, effective and appropriate care
Right to safe health care processes and practices
Right to qualified and competent health workers
Right to safe medical products and their safe and rational use
Right to safe and secure health care facilities
Right to dignity, respect, non-discrimination, privacy and confidentiality
Right to information, education and supported decision making
Right to access to medical records
Right to be heard and fair resolution
Right to patient and family engagement
1
2
3
4
5
6
7
8
9
10
--------------------------------------------------------------------------------

Chunk 3
| 3 |
Dedication
In memory of the patients who have lost their lives or suffered from avoidable harm in health care. May this Charter 
stand as a beacon of hope and commitment, ensuring that the right of every patient to s

In [38]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

print("Embedding model initialized successfully!")

Embedding model initialized successfully!


In [39]:
vectors = embeddings.embed_documents(
    [chunk.page_content for chunk in chunks]
)

print(len(vectors), len(vectors[0]))

55 768


In [40]:
vectorstore = Chroma.from_documents(
    chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

In [41]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

In [42]:
results = retriever.invoke(
    "What are patient safety rights?"
)

for doc in results:
    print(doc.page_content)

Patient safety rights charter
harm in health care to ensure patient safety. These rights recognize that patient safety is impacted by multiple factors, 
such as health workforce management, availability of safe medical products, dignity, respect and non-discrimination, 
information sharing, and patient and family engagement. The rights also acknowledge that patient safety is influenced 
by the socioeconomic environment, the physical environment, and an individual’s personal characteristics; therefore, 
they are formulated with an awareness of the broader context of the determinants of health. 
1. Right to timely, effective and appropriate care. Patients have the right to receive timely and effective 
care tailored to their health needs, particularly in situations where delays in receiving required health care
more. 
In health care settings, patient safety is an important application of human rights norms and standards. 
Right to health. The right to health is the right of everyone to t

In [43]:
system_prompt = SystemMessage(
    content="""
You are an AI Hospital Assistant.
Answer using the hospital knowledge base.
Do not diagnose or recommend medical treatment.
If the answer is not found, say you don't know.
"""
)

memory = InMemorySaver()

In [47]:
from langchain.agents import create_agent
agent = create_agent(
    model="ollama:ornith-1.5:9b",
    system_prompt=system_prompt,
    checkpointer=memory
)

print("Agent created")

Agent created


In [48]:
def search_hospital(query):
    results = retriever.invoke(query)
    return "\n".join(doc.page_content for doc in results)

In [49]:
from langchain_core.tools import tool

@tool
def search_hospital(query: str) -> str:
    """Search hospital documents."""
    results = retriever.invoke(query)
    return "\n".join(doc.page_content for doc in results)

In [50]:
from langchain.agents import create_agent
agent = create_agent(
    model="ollama:ornith-1.5:9b",
    tools=[search_hospital],
    system_prompt=system_prompt,
    checkpointer=memory
)

print("Agent created")

Agent created


In [51]:
config = {
    "configurable": {
        "thread_id": "patient-001"
    }
}

In [52]:
response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="What are patient safety rights?"
            )
        ]
    },
    config=config
)

print(response["messages"][-1].content)

# Patient Safety Rights

Patient safety rights are a set of 10 rights designed to help mitigate potential risks and prevent harm in health care. These rights are shaped by an understanding that patient safety is influenced by many factors, including workforce management, availability of safe medical products, dignity, respect, non-discrimination, information sharing, and engagement of patients and families.

## The 10 Patient Safety Rights

1. **Right to timely, effective and appropriate care** — Patients have the right to receive timely and effective care tailored to their health needs, delivered with compassion and respect.

2. **Right to safe health care processes and practices**

3. **Right to qualified and competent health workers**

4. **Right to safe medical products and their safe and rational use**

5. **Right to safe and secure health care facilities**

6. **Right to dignity, respect, non-discrimination, privacy and confidentiality**

7. **Right to information, education and 